In [ ]:
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.50"

# sanity check
print("PREALLOCATE:", os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"])
print("MEM_FRACTION:", os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"])


import jax, jax.numpy as jnp, time

print("JAX version:", jax.__version__)
print("Default backend:", jax.default_backend())
print("Devices:", jax.devices())

# Pick GPU if available
gpu_devs = jax.devices("gpu")
assert gpu_devs, "No GPU devices visible to JAX."
print("Using device:", gpu_devs[0])

# Simple JIT matmul on GPU
key = jax.random.key(0)
a = jax.random.normal(key, (1024, 1024), dtype=jnp.float32)
b = jax.random.normal(key, (1024, 1024), dtype=jnp.float32)

@jax.jit
def mm(x, y):
    return x @ y

# Warmup compile + run
_ = mm(a, b).block_until_ready()

# Timed run
t0 = time.time()
c = mm(a, b).block_until_ready()
dt = time.time() - t0

print("Result shape:", c.shape)
print("Result device:", c.device)  # <- no parentheses
print(f"JIT matmul ran on {c.device.platform.upper()} in {dt*1e3:.1f} ms")
